# 18 · Choosing an embedding model
Level L3 · Part 3 · builds on lesson 17

Needs: nothing. Everything runs in local mode (`QdrantClient(":memory:")`) on saved vectors in
`data/`. Two cells are optional: one `needs-download` cell loads both models (Qwen3-Embedding-0.6B
is about 1.2 GB) and embeds the test queries; one `needs-download` + `slow` cell embeds the 2,000
recipes with Qwen3 (about 7 minutes on a 2-core CPU). Skipped cells fall back to saved files in
`data/`. The last experiment trains tiny vectors with PyTorch for about 40 seconds.

## Goal
By the end of this lesson you can compare two embedding models on your own test set by quality
and vector size, truncate a Matryoshka model's vectors, and explain the limits of single-vector
embeddings (the LIMIT result).

## The idea
Since lesson 05 every vector came from one small model, all-MiniLM-L6-v2; a public leaderboard,
**MTEB**, ranks hundreds of others. It gives you a shortlist, not the best model for *your*
recipes, so you put each candidate through lesson 16's exam and weigh the score against its cost
in memory and speed. Some models are trained so that the first part of a vector is already a good
vector (**Matryoshka**), so you can cut them short. And with one vector per item, some result
lists cannot be produced at all. Choosing a model is like hiring a cook: the CV gets them an
interview, but you taste their food before you decide.

## Picture

```mermaid
flowchart LR
    MTEB["MTEB leaderboard<br/>shortlist"] --> A["MiniLM<br/>22M params, 384-d"]
    MTEB --> B["Qwen3-Embedding-0.6B<br/>596M params, 1024-d"]
    A --> CA["collection recipes_minilm"]
    B --> CB["collection recipes_qwen3"]
    B --> T["truncate to 512 / 256 / ... / 32<br/>(Matryoshka)"]
    CA & CB & T --> E["lesson 16 exam:<br/>20 labelled queries"]
    E --> D["quality vs<br/>bytes per vector and ms per query"]
```

## Step by step
### 1. The exam and the first candidate
Load the 2,000 recipes, the MiniLM vectors from lesson 14, and the 20 labelled queries from
lesson 16 (`data/eval_queries_20.json`). MiniLM is the model to beat.

In [1]:
import json, math, os, uuid, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore", message="IProgress")
from qdrant_client import QdrantClient, models

recipes = pd.read_csv("../data/recipes_2000.csv")
TITLE = dict(zip(recipes.slug, recipes.title))
qrels = json.load(open("../data/eval_queries_20.json"))["qrels"]
EVAL_QUERIES = list(qrels)
minilm_saved = np.load("../data/recipe_embeddings_minilm_2000.npz")
assert (minilm_saved["slugs"] == recipes.slug.to_numpy()).all()
MINILM = str(minilm_saved["model"])
print(len(recipes), "recipes |", len(qrels), "labelled queries | MiniLM vectors:", minilm_saved["doc_emb"].shape)

2000 recipes | 20 labelled queries | MiniLM vectors: (2000, 384)


### 2. The second candidate, from the leaderboard
**MTEB** (Massive Text Embedding Benchmark) scores embedding models on many tasks: retrieval,
classification, clustering, similarity, reranking and more, in many languages
(<https://huggingface.co/spaces/mteb/leaderboard>). Filter it by what you can run: size, licence,
languages, maximum input length. From it we pick a modern model that still runs on a laptop:
`Qwen/Qwen3-Embedding-0.6B` (Apache 2.0, over 100 languages, vectors of 32 to 1024 dims).
It is about 26 times bigger than MiniLM. The next cell lists the test queries; eight of them are
lesson 16 queries translated into other languages (our own translations), with the same labels.

In [2]:
QWEN = "Qwen/Qwen3-Embedding-0.6B"
TRANSLATED = {   # translated query -> (language, the English query whose labels it shares)
    "un postre de chocolate": ("es", "a chocolate dessert"),
    "sopa fría para un día de calor": ("es", "cold soup for a hot day"),
    "Muscheln in Weißwein": ("de", "mussels in white wine"),
    "un dolce ghiacciato per l'estate": ("it", "a frozen treat for summer"),
    "brochettes d'agneau": ("fr", "lamb skewers"),
    "un ragoût de bœuf mijoté": ("fr", "a slow-cooked beef stew"),
    "भारतीय पनीर करी": ("hi", "Indian cottage cheese curry"),
    "泰式虾仁面条": ("zh", "Thai prawn noodles"),
}
ALL_QUERIES = EVAL_QUERIES + list(TRANSLATED)
QUERY_FILE, DOC_FILE = "../data/query_embeddings_18.npz", "../data/recipe_embeddings_qwen3_2000.npz"
qwen = None
print(len(ALL_QUERIES), "queries:", len(EVAL_QUERIES), "English +", len(TRANSLATED), "translated")

28 queries: 20 English + 8 translated


The download cell loads both models, embeds every query with each, and measures what each model
costs: parameters, vector length, input limit and milliseconds per query on this CPU. Qwen3's
weights ship as bfloat16, which many laptop CPUs compute slowly, so we load them as float32.
Qwen3 is an **instruction** model: `encode_query` puts a task description in front of each query
(`Instruct: Given a web search query, retrieve relevant passages ...`), while documents get none.
We also save the queries embedded *without* it (`encode_document`) to see what it is worth.

In [3]:
os.environ.setdefault("TQDM_DISABLE", "1")
os.environ.setdefault("HF_HUB_VERBOSITY", "error")
import time, torch
from sentence_transformers import SentenceTransformer

minilm = SentenceTransformer(MINILM, device="cpu")
qwen = SentenceTransformer(QWEN, device="cpu", model_kwargs={"dtype": torch.float32})
facts = {}
for name, m in [(MINILM, minilm), (QWEN, qwen)]:
    m.encode_query("warm up")
    start = time.perf_counter()
    for q in EVAL_QUERIES:
        m.encode_query(q)
    facts[name] = {"params": sum(p.numel() for p in m.parameters()), "dim": m.get_embedding_dimension(),
                   "max_tokens": m.max_seq_length, "ms_per_query": 1000 * (time.perf_counter() - start) / len(EVAL_QUERIES)}
np.savez(QUERY_FILE, queries=ALL_QUERIES, facts=json.dumps(facts), minilm=minilm.encode_query(ALL_QUERIES),
         qwen3=qwen.encode_query(ALL_QUERIES), qwen3_no_prompt=qwen.encode_document(ALL_QUERIES))
print("query prompt:", repr(qwen.prompts["query"][:40]), "| saved", len(ALL_QUERIES), "queries per model")

⏭  Skipped: this cell needs [download]. Run it yourself, or re-run with TUT_ENABLE=download.


The slow cell embeds the 2,000 recipes with Qwen3, the same text as MiniLM got
(`title + ". " + description`). It stores them as float16 to keep the file small (4 MB); we turn
them back into float32 when loading.

In [4]:
texts = (recipes.title + ". " + recipes.description).tolist()
start = time.perf_counter()
qwen_docs = qwen.encode_document(texts, batch_size=32)
seconds = time.perf_counter() - start
np.savez_compressed(DOC_FILE, model=QWEN, slugs=recipes.slug.to_numpy(dtype=str),
                    doc_emb=qwen_docs.astype(np.float16), seconds=seconds)
print(f"embedded {len(texts)} recipes in {seconds:.0f} s -> {qwen_docs.shape}")

⏭  Skipped: this cell needs [download,slow]. Run it yourself, or re-run with TUT_ENABLE=download,slow.


### 3. What each candidate costs
Load the saved files and put the facts side by side. Bytes per vector is 4 × dims (float32);
multiply by the number of recipes for the raw vector memory, before any index.

In [5]:
qcache = np.load(QUERY_FILE)
assert qcache["queries"].tolist() == ALL_QUERIES, "query cache is from another query list"
facts = json.loads(str(qcache["facts"]))
qwen_saved = np.load(DOC_FILE)
assert (qwen_saved["slugs"] == recipes.slug.to_numpy()).all() and str(qwen_saved["model"]) == QWEN
DOCS = {"minilm": minilm_saved["doc_emb"], "qwen3": qwen_saved["doc_emb"].astype(np.float32)}
QUERY_VECS = {name: dict(zip(ALL_QUERIES, qcache[name])) for name in ["minilm", "qwen3", "qwen3_no_prompt"]}
cost = pd.DataFrame({short: {"params (M)": facts[full]["params"] / 1e6, "dims": facts[full]["dim"],
                             "max tokens": facts[full]["max_tokens"], "ms / query": facts[full]["ms_per_query"],
                             "bytes / vector": 4 * facts[full]["dim"],
                             "GB for 1M recipes": 4 * facts[full]["dim"] * 1e6 / 1e9}
                     for short, full in [("minilm", MINILM), ("qwen3", QWEN)]}).T
cost = cost.astype({"dims": int, "max tokens": int, "bytes / vector": int})
print(cost.round(1).to_string())
print(f"\nembedding the 2,000 recipes with Qwen3 took {float(qwen_saved['seconds']):.0f} s when the cache was built")

        params (M)  dims  max tokens  ms / query  bytes / vector  GB for 1M recipes
minilm        22.7   384         256        11.3            1536                1.5
qwen3        595.8  1024       32768       398.5            4096                4.1

embedding the 2,000 recipes with Qwen3 took 446 s when the cache was built


These timings were measured on the 2-core CPU that built the cache, so yours will differ;
the ratios are what matter. Qwen3 has 26 times the parameters, its vectors are 2.7 times longer
(4 KB instead of 1.5 KB, so 1M recipes need about 4.1 GB instead of 1.5 GB of raw vectors), and
each query takes about 35 times longer to embed. It also reads up to 32,768 tokens, where MiniLM
stops at 256 (lesson 17). Now: is it better enough to pay for that?

### 4. One collection per model
Each model gets its own collection: vectors from different models cannot share one (lesson 05).
Qdrant 1.16+ lets a collection carry **collection metadata**, a small dictionary of your own.
Recording the model there means any code can check which model a collection expects.

In [6]:
RECIPE_NS = uuid.uuid5(uuid.NAMESPACE_URL, "https://example.com/recipes/")
client = QdrantClient(":memory:")

def make_collection(name, vectors, model_name):
    client.create_collection(name, metadata={"model": model_name, "dim": vectors.shape[1]},
                             vectors_config=models.VectorParams(size=vectors.shape[1], distance=models.Distance.COSINE))
    client.upload_points(name, [
        models.PointStruct(id=str(uuid.uuid5(RECIPE_NS, slug)), vector=vectors[i].tolist(), payload={"slug": slug})
        for i, slug in enumerate(recipes.slug)])
    return client.count(name).count

print("recipes_minilm:", make_collection("recipes_minilm", DOCS["minilm"], MINILM), "points")
print("recipes_qwen3: ", make_collection("recipes_qwen3", DOCS["qwen3"], QWEN), "points")
print("metadata:", client.get_collection("recipes_qwen3").config.metadata)

recipes_minilm: 2000 points


recipes_qwen3:  2000 points
metadata: {'model': 'Qwen/Qwen3-Embedding-0.6B', 'dim': 1024}


### 5. Sit the same exam
Lesson 16's metrics, re-declared, and a `run` that sends a list of query vectors in one
`query_batch_points` call and returns the ranked slugs.

In [7]:
def precision_at_k(ranked, labels, k): return sum(s in labels for s in ranked[:k]) / k
def recall_at_k(ranked, labels, k): return sum(s in labels for s in ranked[:k]) / len(labels)
def reciprocal_rank(ranked, labels): return next((1 / r for r, s in enumerate(ranked, 1) if s in labels), 0.0)

def ndcg_at_k(ranked, labels, k):
    dcg = sum(labels.get(s, 0) / math.log2(r + 1) for r, s in enumerate(ranked[:k], 1))
    ideal = sum(g / math.log2(r + 1) for r, g in enumerate(sorted(labels.values(), reverse=True)[:k], 1))
    return dcg / ideal

def run(collection, vectors, k=10):
    requests = [models.QueryRequest(query=list(map(float, v)), limit=k, with_payload=["slug"]) for v in vectors]
    return [[p.payload["slug"] for p in r.points] for r in client.query_batch_points(collection, requests=requests)]

print(round(ndcg_at_k(["a", "b"], {"a": 2, "b": 1}, 2), 3), reciprocal_rank(["x", "a"], {"a": 1}))

1.0 0.5


`evaluate` runs a model's vectors for a list of queries and scores them against the labels.
A translated query is scored with the labels of the English query it came from.

In [8]:
def labels_for(q):
    return qrels[TRANSLATED[q][1]] if q in TRANSLATED else qrels[q]

def evaluate(collection, query_vecs, queries, k=5, per_query=False):
    ranked = run(collection, [query_vecs[q] for q in queries])
    rows = pd.DataFrame({q: {f"P@{k}": precision_at_k(r, labels_for(q), k), f"R@{k}": recall_at_k(r, labels_for(q), k),
                             "MRR": reciprocal_rank(r, labels_for(q)), f"nDCG@{k}": ndcg_at_k(r, labels_for(q), k)}
                         for q, r in zip(queries, ranked)}).T
    return rows if per_query else rows.mean()

exam = pd.DataFrame({"minilm": evaluate("recipes_minilm", QUERY_VECS["minilm"], EVAL_QUERIES),
                     "qwen3": evaluate("recipes_qwen3", QUERY_VECS["qwen3"], EVAL_QUERIES)}).T
print(exam.round(3).to_string())

         P@5    R@5    MRR  nDCG@5
minilm  0.48  0.556  0.767   0.687
qwen3   0.52  0.680  0.838   0.762


MiniLM reproduces lesson 16's numbers (nDCG@5 0.687). Qwen3 is ahead on every metric. Means hide
a lot, so look query by query: where did the bigger model gain, and where did it lose?

In [9]:
per_query = pd.DataFrame({name: evaluate(f"recipes_{name}", QUERY_VECS[name], EVAL_QUERIES, per_query=True)["nDCG@5"]
                          for name in ["minilm", "qwen3"]})
per_query["change"] = per_query.qwen3 - per_query.minilm
print(per_query.sort_values("change").round(2).to_string())
print("\nbetter:", (per_query.change > 0.05).sum(), "| worse:", (per_query.change < -0.05).sum(),
      "| about the same:", (per_query.change.abs() <= 0.05).sum())

                                     minilm  qwen3  change
a fried Indian pastry snack            1.00   0.63   -0.37
a veggie burger made with beans        1.00   0.84   -0.16
a fish traybake                        1.00   0.85   -0.15
tofu tacos                             1.00   0.87   -0.13
Thai prawn noodles                     1.00   1.00    0.00
a slow-cooked beef stew                1.00   1.00    0.00
a quick vegetarian breakfast           0.00   0.00    0.00
something to dip tortilla chips in     0.00   0.00    0.00
honey ginger duck stir-fry             0.56   0.56    0.00
lamb skewers                           1.00   1.00    0.00
chickpea wrap                          1.00   1.00    0.00
pizza night                            1.00   1.00    0.00
a frozen treat for summer              1.00   1.00    0.00
mussels in white wine                  1.00   1.00    0.00
salmon noodle bowl                     0.85   1.00    0.15
a chocolate dessert                    0.80   0.97    0.

Qwen3 does better on 6 queries and worse on 4; the other 10 are unchanged. Its big wins are
queries that describe a dish instead of naming it. Where does each model rank the dish we want?
Ask both for the top 100:

In [10]:
for q, dish in [("cold soup for a hot day", "gazpacho"), ("eggs cooked in a spicy tomato sauce", "shakshuka")]:
    for name in ["minilm", "qwen3"]:
        ranked = run(f"recipes_{name}", [QUERY_VECS[name][q]], k=100)[0]
        print(f"{name:<7}{q:<37}{dish}: rank", ranked.index(dish) + 1 if dish in ranked else "not in top 100")

minilm cold soup for a hot day              gazpacho: rank 66
qwen3  cold soup for a hot day              gazpacho: rank 1
minilm eggs cooked in a spicy tomato sauce  shakshuka: rank 22
qwen3  eggs cooked in a spicy tomato sauce  shakshuka: rank 4


Qwen3 puts Gazpacho at rank 1 (MiniLM: 66) and Shakshuka at rank 4 (MiniLM:
22), and it lifts "Indian cottage cheese curry" from nDCG@5 0.09 to 0.37. Its losses are on
queries MiniLM already answered perfectly, such as the samosas and the bean burgers. Both still
score 0 on the breakfast and dip queries. A higher mean is not a better answer for every user.

### 6. The instruction matters
Qwen3 was trained with an instruction in front of each query. Leave it out, by embedding queries
like documents, and score again:

In [11]:
no_prompt = evaluate("recipes_qwen3", QUERY_VECS["qwen3_no_prompt"], EVAL_QUERIES)
print(pd.DataFrame({"qwen3 with instruction": exam.loc["qwen3"], "qwen3 without": no_prompt}).T.round(3).to_string())

                         P@5    R@5    MRR  nDCG@5
qwen3 with instruction  0.52  0.680  0.838   0.762
qwen3 without           0.49  0.569  0.757   0.674


Without the instruction Qwen3 drops from nDCG@5 0.762 to 0.674, below MiniLM's 0.687: all
the gain is gone. Read the model card for the prompt each model expects and use `encode_query`
for queries and `encode_document` for documents. MiniLM has empty prompts, so for it the two
are the same (lesson 05).

### 7. Other languages
A **multilingual** model is trained on many languages, so a query and a recipe that mean the same
thing land close together whatever language each is written in. MiniLM was trained on English.
Here are the eight translated queries against the English recipes, top hit for each model:

In [12]:
translated = list(TRANSLATED)
tops = {name: [r[0] for r in run(f"recipes_{name}", [QUERY_VECS[name][q] for q in translated])]
        for name in ["minilm", "qwen3"]}
for i, q in enumerate(translated):
    labels = labels_for(q)
    mark = lambda s: "ok " if s in labels else "-- "
    print(f"{TRANSLATED[q][0]} {q[:30]:<31} MiniLM: {mark(tops['minilm'][i])}{TITLE[tops['minilm'][i]][:24]:<25}"
          f"Qwen3: {mark(tops['qwen3'][i])}{TITLE[tops['qwen3'][i]][:24]}")

es un postre de chocolate          MiniLM: ok Chocolate brownies       Qwen3: ok Chocolate brownies
es sopa fría para un día de calor  MiniLM: -- Spicy halloumi pie       Qwen3: ok Gazpacho
de Muscheln in Weißwein            MiniLM: ok Moules marinieres        Qwen3: ok Moules marinieres
it un dolce ghiacciato per l'esta  MiniLM: -- Honey ginger mushroom pi Qwen3: ok Vanilla ice cream
fr brochettes d'agneau             MiniLM: -- Spicy halloumi skewers   Qwen3: -- French onion soup
fr un ragoût de bœuf mijoté        MiniLM: -- Lasagne                  Qwen3: -- French onion soup
hi भारतीय पनीर करी                 MiniLM: -- Shakshuka                Qwen3: ok Spicy paneer curry
zh 泰式虾仁面条                          MiniLM: -- Kung pao chicken         Qwen3: ok Shrimp pad thai


MiniLM gets two right: the Spanish "chocolate" is the same word in English, and the German
mussels query happens to land on Moules marinieres. On Hindi and Chinese it is lost. Qwen3 gets
six. Its two misses are telling: both French queries return French onion soup. A query in
French lands near things that *look* French, a reminder that language can leak into meaning.

The same eight queries in English and translated, scored by both models:

In [13]:
english = [TRANSLATED[q][1] for q in translated]
langs = pd.DataFrame({f"{name} {which}": evaluate(f"recipes_{name}", QUERY_VECS[name], qs)[["MRR", "nDCG@5"]]
                      for name in ["minilm", "qwen3"] for which, qs in [("English", english), ("translated", translated)]}).T
print(langs.round(3).to_string())

                     MRR  nDCG@5
minilm English     0.792   0.736
minilm translated  0.312   0.270
qwen3 English      1.000   0.917
qwen3 translated   0.750   0.719


MiniLM falls from MRR 0.792 in English to 0.312 when the same questions are translated. Qwen3
falls too, from 1.000 to 0.750, but keeps most of its quality. If your users write in more than
one language, this test matters more than any English leaderboard score.

### 8. Matryoshka: cut the vector short
A **Matryoshka** model (named after the nesting dolls) is trained so the first 512, 256, 128 ...
numbers of its vector are already a usable vector. Cutting a vector to its first `d` numbers is
**truncation**. After cutting, the vector is no longer length 1, so normalise it again (lesson 03).

In [14]:
def truncate(vectors, dims):
    v = np.asarray(vectors, dtype=np.float32)[..., :dims]
    return v / np.linalg.norm(v, axis=-1, keepdims=True)

full = DOCS["qwen3"][0]
print("full:", len(full), "dims, length", round(float(np.linalg.norm(full)), 3))
print("first 256:", round(float(np.linalg.norm(full[:256])), 3), "-> after truncate():",
      round(float(np.linalg.norm(truncate(full, 256))), 3))

full: 1024 dims, length 1.0
first 256: 0.49 -> after truncate(): 1.0


Score every length on the exam. Each one gets its own collection, built from the cut recipe
vectors and queried with the cut query vectors. We do the same to MiniLM, which was **not** trained
this way, to see the difference.

In [15]:
def truncation_curve(name, dims_list):
    rows = {}
    for dims in dims_list:
        coll = f"{name}_{dims}"
        make_collection(coll, truncate(DOCS[name], dims), MINILM if name == "minilm" else QWEN)
        qv = {q: truncate(QUERY_VECS[name][q], dims) for q in EVAL_QUERIES}
        rows[dims] = {"bytes / vector": 4 * dims, **evaluate(coll, qv, EVAL_QUERIES)[["MRR", "nDCG@5"]]}
        client.delete_collection(coll)
    return pd.DataFrame(rows).T.astype({"bytes / vector": int}).rename_axis("dims")

qwen_curve = truncation_curve("qwen3", [1024, 512, 256, 128, 64, 32])
print(qwen_curve.round(3).to_string())

      bytes / vector    MRR  nDCG@5
dims                               
1024            4096  0.838   0.762
512             2048  0.777   0.725
256             1024  0.747   0.673
128              512  0.705   0.619
64               256  0.625   0.454
32               128  0.412   0.304


Halving Qwen3 to 512 dims costs 0.04 nDCG@5 (0.762 to 0.725), still above full MiniLM (0.687)
with a third more bytes. At 256 dims (1 KB a vector) it scores 0.673, about MiniLM's quality in
two thirds of MiniLM's memory. Below 128 dims the quality falls off fast.

The same cuts for MiniLM:

In [16]:
minilm_curve = truncation_curve("minilm", [384, 192, 96, 48])
print(minilm_curve.round(3).to_string())
keep = lambda curve: (curve["nDCG@5"] / curve["nDCG@5"].iloc[0]).round(2).tolist()
print("\nshare of full nDCG@5 kept  qwen3:", keep(qwen_curve), " minilm:", keep(minilm_curve))

      bytes / vector    MRR  nDCG@5
dims                               
384             1536  0.767   0.687
192              768  0.775   0.674
96               384  0.697   0.589
48               192  0.683   0.528

share of full nDCG@5 kept  qwen3: [1.0, 0.95, 0.88, 0.81, 0.6, 0.4]  minilm: [1.0, 0.98, 0.86, 0.77]


MiniLM loses about as large a share when cut to a half, a quarter, an eighth (0.98, 0.86, 0.77
against Qwen3's 0.95, 0.88, 0.81). On this 20-query exam we cannot see a Matryoshka advantage:
MiniLM keeps a similar share (better at a half, slightly worse at a quarter and an eighth). Larger
benchmarks do measure a small edge (sentence-transformers docs: 98.37% vs 96.46% kept at 8.3% of
the dims). What matters is that Qwen3 *starts* higher and offers sizes
from 32 to 1024 as a supported option. Cutting a model that was not trained for it is a gamble
you must measure first, as here.

### 9. Store the short vectors
Truncation happens before the database sees the vectors: the collection is simply created with
the shorter size, and every query must be cut the same way. Sentence-transformers can do it for
you with `SentenceTransformer(..., truncate_dim=256)` or `encode_query(..., truncate_dim=256)`
(it cuts but does not normalise again; a Cosine collection normalises on upload anyway).
Record the length in the metadata too.

In [17]:
make_collection("recipes_qwen3_256", truncate(DOCS["qwen3"], 256), QWEN)
client.update_collection("recipes_qwen3_256", metadata={"truncate_dim": 256})
print("metadata:", client.get_collection("recipes_qwen3_256").config.metadata)
q = "a frozen treat for summer"
hits = client.query_points("recipes_qwen3_256", query=truncate(QUERY_VECS["qwen3"][q], 256).tolist(), limit=3).points
print(q, "->", [(TITLE[h.payload["slug"]], round(h.score, 3)) for h in hits])

metadata: {'model': 'Qwen/Qwen3-Embedding-0.6B', 'dim': 256, 'truncate_dim': 256}
a frozen treat for summer -> [('Lemon sorbet', 0.729), ('Vanilla ice cream', 0.604), ('Panna cotta', 0.588)]


`update_collection(metadata=...)` adds keys and keeps the existing ones. A quarter of the memory,
and the summer query still finds the two frozen desserts it is labelled with.

### 10. What no model can do: the LIMIT result
Every model so far turns a recipe into **one** vector and ranks by one score. Weller et al.
(ICLR 2026, <https://arxiv.org/abs/2508.21038>) proved that this has a hard limit set by the
dimension: with `d` dims, only so many different top-k lists can ever be produced, whatever
the training. Picture queries like "recipes with both basil and lime": every *pair* of
ingredients is a different top-2 list. With enough items, some pairs cannot be anyone's top 2.

We can test this the way the paper did, with **free embeddings**: no model at all, just vectors we
are allowed to move anywhere. Give `n` documents and one query per pair of documents their own
`d`-dim vectors and optimise them directly so each query's top 2 is exactly its pair. If even
that fails, no trained model of that size can succeed.

How the training loop below works, in plain words. Every query is scored against every document.
The **loss** is one number that gets smaller when each query's two right documents have the
highest scores (dividing the scores by 0.05 makes the gaps between them sharper). At each step the
**optimiser** (Adam, from PyTorch) nudges every vector a little in the direction that lowers the
loss. Every 100 steps we check whether every query's top 2 is exactly its pair, and stop as soon
as it is.

In [18]:
import itertools, torch
import torch.nn.functional as F

def all_pairs_solvable(n, d, steps=2000, seed=0):
    torch.manual_seed(seed)
    pairs = torch.tensor(list(itertools.combinations(range(n), 2)))
    docs, queries = torch.randn(n, d, requires_grad=True), torch.randn(len(pairs), d, requires_grad=True)
    target = torch.zeros(len(pairs), n)
    target[torch.arange(len(pairs))[:, None], pairs] = 0.5         # half the weight on each doc of the pair
    opt = torch.optim.Adam([docs, queries], lr=0.05)
    for step in range(steps):
        scores = F.normalize(queries, dim=1) @ F.normalize(docs, dim=1).T
        loss = -(target * F.log_softmax(scores / 0.05, dim=1)).sum(1).mean()   # small when the pair scores highest
        opt.zero_grad(); loss.backward(); opt.step()
        if step % 100 == 99:
            top2 = scores.detach().topk(2, dim=1).indices.sort(1).values
            if (top2 == pairs).all():
                return True
    return False

def solvable(n, d):                                                 # two random starts
    return all_pairs_solvable(n, d, seed=0) or all_pairs_solvable(n, d, seed=1)

print("4 docs in 2 dims:", solvable(4, 2), "| 4 docs in 3 dims:", solvable(4, 3))

4 docs in 2 dims: False | 4 docs in 3 dims: True


In 2 dims, unit vectors sit on a circle, and the top 2 for any direction are always two
*neighbouring* points. With 4 points, the two pairs that sit across from each other are never
neighbours, so they can never be a top 2. In 3 dims all six
pairs work. The optimiser can get stuck in a bad start, so we give it two random starts.

Now find, for each dimension, the most documents whose every pair can be returned. Add one
document at a time until `solvable` fails. This takes about 40 s.

In [19]:
critical = {}
for d in [3, 4, 5, 6]:
    n = d + 1
    while solvable(n, d):
        n += 1
    critical[d] = n - 1
paper_fit = lambda d: -10.5322 + 4.0309 * d + 0.0520 * d**2 + 0.0037 * d**3
print("dims -> most docs with every pair as some query's top 2:", critical)
print("paper's fit extrapolated: 384 dims ->", f"{paper_fit(384):,.0f} docs,", "1024 dims ->", f"{paper_fit(1024):,.0f} docs")

dims -> most docs with every pair as some query's top 2: {3: 4, 4: 10, 5: 16, 6: 27}
paper's fit extrapolated: 384 dims -> 218,711 docs, 1024 dims -> 4,031,488 docs


The limit grows with the dimension: 4 documents in 3 dims, 10 in 4 dims (the paper also finds 10;
its Table 6 starts at d=4), then 16 and 27 in 5 and 6 dims, where the paper reports 14 and 19.
Both are lower bounds from an optimiser, so treat the extrapolation as a rough order of magnitude.
The paper fitted a curve to d = 4 to 45; extrapolated, it gives about 219 thousand documents for
384 dims and about 4 million for 1024 (the paper itself quotes 4m for 1024). And this is with
vectors chosen *for* the test; a real model, trained on other text, does worse.

The paper then built **LIMIT**, a plain-language dataset with exactly this structure (50,000
documents like "Jon likes Hawaiian pizza", 1,000 queries like "who likes Hawaiian pizza?" with 2
right answers each). State-of-the-art single-vector models found fewer than 20% of the right
answers in their top 100, while BM25 keyword search found 93.6% and a multi-vector model
(GTE-ModernColBERT) found 54.8%, better than any single-vector model but far from solved (paper,
Table 5). The paper's own Qwen3 Embed, the family used in this lesson, found only 4.8%. The lesson for the recipe finder:
when queries combine many independent facts, don't hope a bigger model fixes it. Use payload
filters for hard facts (lesson 11), keyword or sparse vectors (lesson 19) and several vectors per
item (lessons 22-23).

## What just happened
- Each model got its own collection, named in its metadata; Qwen3 beat MiniLM on nDCG@5 (0.762 vs 0.687) and MRR (0.838 vs 0.767), yet won 6, lost 4, tied 10 queries.
- The price: 26 times the parameters, 4 KB vs 1.5 KB per vector, about 400 vs 11 ms per query.
- Without its query instruction Qwen3 fell to 0.674; on translated queries MiniLM's MRR fell to 0.312, Qwen3 kept 0.750.
- Qwen3 cut to 512 dims still beat full MiniLM, and at 256 dims matched it in two thirds of the memory; MiniLM lost a similar share when cut.
- Free embeddings showed the dimension limit: every pair fits as a top 2 for only 4, 10, 16, 27 documents in 3–6 dims.

## Common mistakes

**1. Forgetting to cut the query.** The collection holds 256-dim vectors; the query came straight
from the model with 1024:

In [20]:
try:
    client.query_points("recipes_qwen3_256", query=QUERY_VECS["qwen3"]["lamb skewers"].tolist(), limit=3)
except Exception as err:  # this mistake is on purpose: show the error, keep the notebook running
    print(f"{type(err).__name__}: {err}")

ValueError: shapes (2000,256) and (1024,) not aligned: 256 (dim 1) != 1024 (dim 0)


It prints `ValueError: shapes (2000,256) and (1024,) not aligned: ...`: local mode compares the
query with every stored vector in NumPy and the shapes don't fit. A server rejects a query of the
wrong length as well. That one is loud. Fix: put the cut in one function that every query goes through, and read the length from
the collection, not from a constant typed elsewhere:

In [21]:
def embed_for(collection, query, query_vecs):
    dims = client.get_collection(collection).config.params.vectors.size
    return truncate(query_vecs[query], dims).tolist()

hits = client.query_points("recipes_qwen3_256", query=embed_for("recipes_qwen3_256", "lamb skewers", QUERY_VECS["qwen3"]), limit=3).points
print([TITLE[h.payload["slug"]] for h in hits])

['Herby lamb skewers', 'Creamy lamb skewers', 'Lemon garlic lamb skewers']


**2. A query from the wrong model, with the right length.** MiniLM gives 384 numbers. Cut Qwen3
to 384 and the lengths match, so Qdrant accepts a MiniLM query without complaint, and returns
nonsense:

In [22]:
try:
    make_collection("recipes_qwen3_384", truncate(DOCS["qwen3"], 384), QWEN)
    wrong = {q: QUERY_VECS["minilm"][q] for q in EVAL_QUERIES}          # MiniLM queries, Qwen3 recipes
    score = evaluate("recipes_qwen3_384", wrong, EVAL_QUERIES)["nDCG@5"]
    assert score > 0.5, f"nDCG@5 = {score:.3f}: queries and recipes come from different models"
except Exception as err:  # this mistake is on purpose: show the error, keep the notebook running
    print(f"{type(err).__name__}: {err}")

AssertionError: nDCG@5 = 0.000: queries and recipes come from different models


It prints `AssertionError: nDCG@5 = ...: queries and recipes come from different models`. No error
from the database: both are lists of 384 floats. Fix: store the model in the collection metadata
(step 4) and check it before every query:

In [23]:
def check_model(collection, query_model):
    stored = client.get_collection(collection).config.metadata["model"]
    if stored != query_model:
        raise ValueError(f"{collection!r} holds {stored} vectors; the query came from {query_model}")
    return stored

print("ok:", check_model("recipes_qwen3", QWEN))

ok: Qwen/Qwen3-Embedding-0.6B


## Try it
1. **Pick the dimension.** Find the smallest Qwen3 truncation whose mean nDCG@5 is within 0.02
   of the full 1024-dim score (try lengths between those in step 8 too), and work out the raw
   vector memory for 1 million recipes at that length. Is it smaller than MiniLM's 1.5 GB?
2. **Does the multilingual skill survive the cut?** Score the eight translated queries against
   Qwen3 truncated to 256 and to 64 dims. Which loses more when cut, English or translated?

In [24]:
# your code here

Clean up: drop the throwaway collection from mistake 2 and close the local client.

In [25]:
client.delete_collection("recipes_qwen3_384")
client.close()
print("client closed")

client closed


## Key terms
- **MTEB** — Massive Text Embedding Benchmark: a public leaderboard scoring embedding models on many tasks and languages; a source for a shortlist, not a final answer.
- **model size** — the number of parameters (weights) in a model; bigger usually means better vectors and slower, costlier embedding.
- **instruction (query prompt)** — a task description a model like Qwen3 expects in front of each query; `encode_query` adds it, documents get none.
- **multilingual embeddings** — vectors from a model trained on many languages, so a query and a document that mean the same land close together across languages.
- **Matryoshka embeddings** — vectors trained so that their first d numbers are already a good shorter vector.
- **truncation** — keeping only the first d numbers of a vector (then normalising again) to save memory and time.

- **collection metadata** — a dictionary of your own stored on a Qdrant collection (1.16+), e.g. the embedding model and dimension.
- **free embeddings** — vectors optimised directly for a test, with no model; the best any model of that dimension could do.
- **loss** — one number that measures how wrong the current vectors are for the goal; training tries to make it smaller.
- **optimiser** — the routine (here Adam, from PyTorch) that nudges the numbers being trained a little at each step so the loss goes down.
- **embedding-dimension limit** — with d dims, a single vector per item can only produce a limited number of different top-k lists (LIMIT, ICLR 2026).